# 12 Validation, Stats, and Reporting

This notebook audits the higher-level quality checks around a capture: `tl.validate` for forward and backward parity, `tl.aggregate` with streaming `Norm` statistics over a dataloader-like loop, and readable reports from `tl.report.explain` plus `Trace.summary()`.

The model is scalar-friendly for backward validation, but still has enough intermediate structure for reports and aggregation.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "audit":
    REPO_ROOT = REPO_ROOT.parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"repo root: {REPO_ROOT}")

In [ ]:
import torch
from torch import nn
import torchlens as tl

torch.manual_seed(53)


class ReportingNet(nn.Module):
    """Tiny model for validation and reporting examples."""

    def __init__(self) -> None:
        """Initialize layers."""

        super().__init__()
        self.hidden = nn.Linear(3, 4)
        self.output = nn.Linear(4, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run a small regression head.

        Parameters
        ----------
        x:
            Input tensor with three features.

        Returns
        -------
        torch.Tensor
            One value per row.
        """

        return self.output(torch.relu(self.hidden(x)))


def loss_fn(output: torch.Tensor) -> torch.Tensor:
    """Return a scalar loss for backward validation.

    Parameters
    ----------
    output:
        Model output tensor.

    Returns
    -------
    torch.Tensor
        Scalar loss.
    """

    return output.sum()


model = ReportingNet().eval()
x = torch.randn(2, 3)
trace = tl.trace(model, x, layers_to_save="all")
print(f"layers: {[layer.layer_label for layer in trace.layers]}")
print(f"output layer: {trace.output_layers[-1]}")

`tl.validate(..., scope="forward")` replays the forward capture path. `scope="backward"` compares TorchLens backward capture with ordinary PyTorch autograd.

In [ ]:
forward_ok = tl.validate(
    model,
    x,
    scope="forward",
    random_seed=53,
    validate_metadata=False,
)
backward_ok = tl.validate(
    model,
    x,
    scope="backward",
    loss_fn=loss_fn,
    random_seed=53,
    validate_metadata=False,
)
print(f"forward parity: {forward_ok}")
print(f"backward parity: {backward_ok}")

> NOTE: The executable `tl.validate(..., scope="saved")` path currently delegates to the same saved-forward validator as `scope="forward"`. This notebook uses the names and behavior that run.

`tl.aggregate` streams traces over an iterable of batches. Here `Norm` stores the mean output L2 norm without keeping every activation from every batch.

In [ ]:
dataloader = [torch.randn(2, 3) + offset for offset in (0.0, 0.5, 1.0)]
output_stats = tl.aggregate(
    model,
    dataloader,
    metrics={"output": tl.stats.Norm(name="output_l2")},
)
print(output_stats)

The same interface can stream gradient statistics when given `target="grad"` and a loss function. The metric key can be a layer label.

In [ ]:
hidden_label = next(layer.layer_label for layer in trace.layers if layer.func_name == "linear")
grad_stats = tl.aggregate(
    model,
    [(batch, torch.zeros(batch.shape[0], 1)) for batch in dataloader],
    metrics={hidden_label: tl.stats.Norm(name="hidden_grad_l2")},
    target="grad",
    loss_fn=lambda output, target: ((output - target) ** 2).mean(),
)
print(f"gradient metric layer: {hidden_label}")
print(grad_stats)

`tl.report.explain` produces a plain-language report for a completed trace. `summary()` remains the compact table view.

In [ ]:
report_text = tl.report.explain(trace, audience="practitioner")
print("\n".join(report_text.splitlines()[:12]))
print("\n--- summary ---")
print(trace.summary(fields=["name", "shape", "params"]))

Reports are strings, so they can be saved, diffed, or attached to experiment output without requiring a notebook renderer.

In [ ]:
research_report = tl.report.explain(trace, audience="researcher")
print(f"research report lines: {len(research_report.splitlines())}")
print(f"mentions anomalies: {'Anomalies' in research_report}")

## Validation Diagnostics and Extra Metrics
Validation returns booleans for supported scopes. This build does not currently surface a forced forward-failure diagnostic for a state-mutating toy model, so the probe below is explicitly labeled as a validation diagnostic gap rather than presented as a failing example. The cell still probes saved/intervention scopes and adds metrics beyond `tl.stats.Norm`.


In [ ]:
class StatefulFailNet(nn.Module):
    """Model that mutates state between calls for validation-gap probing."""

    def __init__(self) -> None:
        """Initialize a call counter buffer."""
        super().__init__()
        self.register_buffer("counter", torch.zeros(()))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Increment the counter and add it to the output."""
        self.counter += 1
        return x + self.counter


try:
    stateful_probe = tl.validate(
        StatefulFailNet().eval(),
        torch.ones(2, 3),
        scope="forward",
        random_seed=53,
        validate_metadata=False,
        verbose=True,
    )
    print(
        "> NOTE: validation forced-failure diagnostic gap: this build's tl.validate "
        "does not surface a failure for the state-mutating StatefulFailNet probe."
    )
    print(f"stateful probe returned: {stateful_probe}")
except Exception as exc:
    print(f"validation forced-failure diagnostic surfaced: {type(exc).__name__}: {exc}")

for scope in ["saved", "intervention"]:
    try:
        result = tl.validate(
            model,
            x,
            scope=scope,
            random_seed=53,
            validate_metadata=False,
        )
        print(f"scope={scope!r} validation: {result}")
    except Exception as exc:
        print(f"> NOTE: scope={scope!r} validation gap: {type(exc).__name__}: {exc}")

extra_stats = tl.aggregate(
    model,
    dataloader,
    metrics={
        "output": tl.stats.Mean(name="output_mean"),
        hidden_label: tl.stats.Quantile((0.25, 0.5, 0.75), name="hidden_quantiles"),
    },
)
print(extra_stats)
try:
    print(tl.report.explain(trace, audience="executive").splitlines()[0])
except ValueError as exc:
    print(f"> NOTE: report audience gap: {exc}")
print(tl.report.explain(trace, audience="auto").splitlines()[0])

## 🔧 Sandbox

Mini-experiment: change `sandbox_p`, `sandbox_values`, and the metric name. Expected observation: the aggregate norm changes as the input batches change.

In [ ]:
sandbox_p = 2.0
sandbox_values = (0.0, 1.0, 2.0)
sandbox_metric_name = "sandbox_output_norm"
sandbox_batches = [torch.full((1, 3), value) for value in sandbox_values]
baseline_stats = tl.aggregate(
    model,
    [x.detach()],
    metrics={"output": tl.stats.Norm(p=sandbox_p, name=sandbox_metric_name)},
)
sandbox_stats = tl.aggregate(
    model,
    sandbox_batches,
    metrics={"output": tl.stats.Norm(p=sandbox_p, name=sandbox_metric_name)},
)
print("baseline:")
print(baseline_stats)
print("sandbox:")
print(sandbox_stats)